In [1]:
import os
from datasets import load_dataset
from tqdm import tqdm

# 1. Setup paths
os.makedirs("data/pretrain", exist_ok=True)
output_path = "data/pretrain/data.txt"

# 2. Targeted Saudi Vision 2030 injection (The Base Knowledge)
saudi_context = """
Saudi Vision 2030 is the roadmap for the Kingdom's future, focusing on a vibrant society and a thriving economy.
The Saudi Data and AI Authority (SDAIA) leads the national agenda for Data and Artificial Intelligence.
Digital transformation is a key driver for government efficiency and citizen services in Saudi Arabia.
NEOM and The Line represent the pinnacle of AI-driven urban planning and cognitive cities.
Innovation, Cloud Computing, and Cybersecurity are the backbones of the Kingdom's digital economy.
"""

# 3. Load a reliable Technical Dataset (Stack Exchange Academic/Tech)
print("📥 Loading Tech-Focused Dataset (Academic Stack Exchange)...")
try:
    # This dataset contains high-quality technical Q&A across many domains
    ds = load_dataset("freehold/academic_stack_exchange", split="train", streaming=True)
except Exception as e:
    print(f"Error: {e}. Falling back to a standard tech corpus...")
    ds = load_dataset("tatsu-lab/alpaca", split="train", streaming=True)

# 4. Curate and Save
print("🔍 Filtering for pure Technology and Science...")
collected_text = [saudi_context]
current_size = len(saudi_context.encode('utf-8'))
max_size = 3 * 1024 * 1024  # 3MB is perfect for an M1 pre-training demo

# We look for tech-heavy articles
tech_keywords = ["computer", "algorithm", "data", "software", "network", "system", "logic"]

for entry in tqdm(ds):
    # Depending on dataset, field might be 'text', 'instruction', or 'content'
    text = entry.get('text', entry.get('content', ""))
    
    if len(text) > 200 and any(k in text.lower() for k in tech_keywords):
        collected_text.append(text)
        current_size += len(text.encode('utf-8'))
    
    if current_size >= max_size:
        break

# 5. Final Write
with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n\n---\n\n".join(collected_text))

print(f"\n✅ SUCCESS!")
print(f"📊 Final Size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")
print(f"📍 Data ready at: {output_path}")

📥 Loading Tech-Focused Dataset (Academic Stack Exchange)...
Error: Dataset 'freehold/academic_stack_exchange' doesn't exist on the Hub or cannot be accessed.. Falling back to a standard tech corpus...
🔍 Filtering for pure Technology and Science...


28791it [00:09, 2880.96it/s] 



✅ SUCCESS!
📊 Final Size: 3.03 MB
📍 Data ready at: data/pretrain/data.txt


In [14]:
import os

input_file = 'data/pretrain/data.txt'
output_file = 'data/pretrain/data_clean.txt'

# Labels to remove
tags_to_strip = [
    "### Instruction:", 
    "### Input:", 
    "### Response:", 
    "Below is an instruction that describes a task", 
    "paired with an input that provides further context.", 
    "Write a response that appropriately completes the request."
]

cleaned_lines = []

if os.path.exists(input_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            clean_line = line
            # Remove the structural tags
            for tag in tags_to_strip:
                clean_line = clean_line.replace(tag, "")
            
            # Keep the line if it actually contains information (not just whitespace)
            if clean_line.strip():
                cleaned_lines.append(clean_line.strip())

    # Save the cleaned version
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("\n".join(cleaned_lines))
    
    # Swap them so the training script uses the clean one
    os.rename(output_file, input_file)
    print(f"✅ Data Purified! {len(cleaned_lines)} blocks of knowledge preserved.")
else:
    print("❌ Could not find data/pretrain/data.txt")

✅ Data Purified! 26748 blocks of knowledge preserved.


In [15]:
import os
import json
import re
from collections import Counter

class BPETokenizer:
    def __init__(self, vocab_size=500):
        self.vocab_size = vocab_size
        self.merges = {}
        self.vocab = {}
        self.inverse_vocab = {}

    def get_stats(self, ids):
        counts = Counter()
        for i in range(len(ids) - 1):
            counts[(ids[i], ids[i+1])] += 1
        return counts

    def merge(self, ids, pair, idx):
        new_ids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
                new_ids.append(idx)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        return new_ids

    def train(self, file_path):
        print(f"Training BPE Tokenizer on {file_path}...")
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        # Start with individual bytes (0-255)
        tokens = list(text.encode("utf-8"))
        
        # Iteratively merge the most frequent pairs
        num_merges = self.vocab_size - 256
        for i in range(num_merges):
            stats = self.get_stats(tokens)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            tokens = self.merge(tokens, pair, idx)
            self.merges[pair] = idx
            print(f"Merge {i+1}/{num_merges}: {pair} -> {idx}", end="\r")

        # Create vocabulary mapping
        self.vocab = {i: bytes([i]) for i in range(256)}
        for (p0, p1), idx in self.merges.items():
            self.vocab[idx] = self.vocab[p0] + self.vocab[p1]
        
        self.inverse_vocab = {v: k for k, v in self.vocab.items()}
        print("\n✅ Tokenizer training complete!")

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        while len(tokens) >= 2:
            stats = self.get_stats(tokens)
            # Find the first merge that applies
            pair = min(stats.keys(), key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            tokens = self.merge(tokens, pair, self.merges[pair])
        return tokens

    def decode(self, ids):
        tokens = b"".join(self.vocab[idx] for idx in ids)
        return tokens.decode("utf-8", errors="replace")

    def save(self, path):
        # We save merges as strings because JSON keys must be strings
        data = {
            "merges": {f"{k[0]},{k[1]}": v for k, v in self.merges.items()},
            "vocab": {k: v.decode("latin-1") for k, v in self.vocab.items()}
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"Saved tokenizer to {path}")

# --- Test the Tokenizer ---
if __name__ == "__main__":
    tokenizer = BPETokenizer(vocab_size=600)
    tokenizer.train("data/pretrain/data.txt")
    
    test_str = "SDAIA leads the way in Vision 2030."
    encoded = tokenizer.encode(test_str)
    decoded = tokenizer.decode(encoded)
    
    print(f"\nTest String: {test_str}")
    print(f"Encoded IDs: {encoded}")
    print(f"Decoded: {decoded}")
    
    tokenizer.save("src/tokenizer/vocab.json")

Training BPE Tokenizer on data/pretrain/data.txt...
Merge 344/344: (263, 290) -> 599
✅ Tokenizer training complete!

Test String: SDAIA leads the way in Vision 2030.
Encoded IDs: [83, 68, 451, 65, 32, 317, 402, 257, 272, 119, 594, 302, 86, 597, 296, 50, 48, 51, 48, 46]
Decoded: SDAIA leads the way in Vision 2030.
Saved tokenizer to src/tokenizer/vocab.json


In [17]:
import json
import os
import random

# 1. Setup paths
os.makedirs("data/finetune", exist_ok=True)
output_path = "data/finetune/instruction-data.json"

# 2. Define our "Pure Knowledge" Base
saudi_tech_knowledge = [
    {
        "topic": "SDAIA",
        "explanation": "SDAIA stands for the Saudi Data and AI Authority. Think of it as the 'brain' of the Kingdom's digital future. They make sure the country uses data and robots safely and smartly to help people.",
        "keywords": ["SDAIA", "Authority", "Data", "AI Agency"]
    },
    {
        "topic": "Vision 2030",
        "explanation": "Vision 2030 is Saudi Arabia's big plan for the future. It's like a map that shows how the country will grow its economy beyond oil, using technology, tourism, and smart cities.",
        "keywords": ["Vision 2030", "The Plan", "Future of Saudi", "Diversification"]
    },
    {
        "topic": "Artificial Intelligence",
        "explanation": "AI is when we teach computers to learn and solve problems like humans do. For example, it helps cars drive themselves or helps doctors find diseases faster.",
        "keywords": ["AI", "Artificial Intelligence", "Machine Learning", "Smart Computers"]
    },
    {
        "topic": "The Line (NEOM)",
        "explanation": "The Line is a city being built in NEOM that is a straight line. It has no cars or streets. Everything you need is a 5-minute walk away, and AI helps run the city behind the scenes.",
        "keywords": ["The Line", "NEOM City", "Linear City", "No Cars"]
    },
    {
        "topic": "Cloud Computing",
        "explanation": "Instead of saving your photos on one phone, Cloud Computing saves them on a giant network of computers far away. You can access your stuff from anywhere in the world.",
        "keywords": ["Cloud", "Internet Storage", "Remote Servers"]
    },
    {
        "topic": "Cybersecurity",
        "explanation": "Cybersecurity is like a digital lock and a security guard for your computer. It stops bad people from stealing your private information or breaking your apps.",
        "keywords": ["Cybersecurity", "Digital Safety", "Hacking Protection"]
    }
]

# 3. Instruction Templates (To create variety)
templates = [
    "What is {}?",
    "Can you explain {} so a child can understand?",
    "Why is {} important for Saudi Arabia's future?",
    "Give me a simple summary of {}.",
    "How does {} help people in their daily lives?",
    "Tell me about {} and its role in Vision 2030."
]

# 4. Synthesize the "Huge" Dataset
final_dataset = []

# Generate 1,500 entries by varying the templates and topics
for _ in range(250): # Repeat to create 1,500 variations
    for item in saudi_tech_knowledge:
        topic = item['topic']
        instr = random.choice(templates).format(topic)
        
        final_dataset.append({
            "instruction": instr,
            "input": "",
            "output": item['explanation']
        })

# 5. Save the clean, fluff-free dataset
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=4, ensure_ascii=False)

print(f"✅ SUCCESS! Created {len(final_dataset)} PURE tech/Saudi Q&A pairs.")
print(f"📍 Location: {output_path}")
print(f"📊 Preview: {final_dataset[0]['instruction']} -> {final_dataset[0]['output'][:50]}...")

✅ SUCCESS! Created 1500 PURE tech/Saudi Q&A pairs.
📍 Location: data/finetune/instruction-data.json
📊 Preview: What is SDAIA? -> SDAIA stands for the Saudi Data and AI Authority. ...


In [18]:
%%writefile /Users/layooni/Documents/gpt-from-scratch/src/model/transformer.py
import torch
import torch.nn as nn
from torch.nn import functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0
        self.key = nn.Linear(n_embd, n_embd, bias=False)
        self.query = nn.Linear(n_embd, n_embd, bias=False)
        self.value = nn.Linear(n_embd, n_embd, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.proj = nn.Linear(n_embd, n_embd)
        self.n_head = n_head

    def forward(self, x):
        B, T, C = x.shape
        # Split Q, K, V into multiple heads
        k = self.key(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = self.query(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = self.value(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # Causal self-attention
        wei = q @ k.transpose(-2, -1) * (C // self.n_head)**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        
        out = wei @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size)
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class DaleelGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=4, block_size=128):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.block_size = block_size

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

Overwriting /Users/layooni/Documents/gpt-from-scratch/src/model/transformer.py


In [22]:
import torch
import torch.optim as optim

# 1. Hyperparameters for a smooth M1 experience
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 32 
block_size = 128
max_iters = 2000
eval_interval = 200
learning_rate = 1e-3 # Slightly higher for faster initial convergence

# 2. Data Preparation (using the data.txt we built)
with open('data/pretrain/data.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
data = torch.tensor(encode(text), dtype=torch.long)

def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# 3. Model Initialization
model = DaleelGPT(vocab_size=vocab_size).to(device)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

print(f"🚀 Training Daleel on {device.upper()}...")

for iter in range(max_iters):
    if iter % eval_interval == 0:
        model.eval()
        x_val, y_val = get_batch()
        _, loss = model(x_val, y_val)
        print(f"Step {iter}: Loss {loss.item():.4f}")
        model.train()

    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# 4. Save the Base Weights
torch.save(model.state_dict(), "models/daleel_base.pt")
print("✅ Pre-training complete! Weights saved to models/daleel_base.pt")

🚀 Training Daleel on MPS...
Step 0: Loss 4.9611
Step 200: Loss 2.4349
Step 400: Loss 2.1665
Step 600: Loss 1.8744
Step 800: Loss 1.5053
Step 1000: Loss 1.4578
Step 1200: Loss 1.2998
Step 1400: Loss 1.3715
Step 1600: Loss 1.2721
Step 1800: Loss 1.2035
✅ Pre-training complete! Weights saved to models/daleel_base.pt


In [31]:
import torch
import json
import random
from transformer import DaleelGPT

# 1. Setup
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

# Reload vocab (crucial for consistency)
with open('data/pretrain/data.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]

# 2. Load Base Model & UNFREEZE
model = DaleelGPT(vocab_size=vocab_size).to(device)
model.load_state_dict(torch.load("models/daleel_base.pt", weights_only=True))

# FORCE UNFREEZE: Ensure all parameters are trainable
for param in model.parameters():
    param.requires_grad = True

model.train() # Set to training mode

# 3. Load Instruction Data
with open('data/finetune/instruction-data.json', 'r') as f:
    fine_tune_data = json.load(f)

# 4. Optimizer - Use a slightly higher LR to "break" the plateau
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4) 
steps = 1500 # A few more steps to ensure format learning
block_size = 128

print("🎯 Re-starting Fine-Tuning (Fixing the Gradient Flow)...")

for i in range(steps):
    item = random.choice(fine_tune_data)
    # The template is key for the model to recognize instructions
    full_text = f"### Instruction:\n{item['instruction']}\n\n### Response:\n{item['output']}"
    
    tokens = encode(full_text)
    if len(tokens) > block_size:
        tokens = tokens[-block_size:]
    if len(tokens) > 1:
        # Sliding window for the instruction-response pair
        idx = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
        
        # We only want to train on the response part usually, 
        # but for a small model, training on the whole sequence is fine.
        x = idx[:, :-1]
        y = idx[:, 1:]
        
        logits, loss = model(x, y)
        
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        
        # Clip gradients to prevent "exploding" (common in SFT)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()

    if i % 100 == 0:
        print(f"Step {i}: Loss = {loss.item():.4f}")

# 5. Final Save
torch.save(model.state_dict(), "models/daleel_final.pt")
print("🚀 SUCCESS! Check the loss numbers above—they should be changing now.")

🎯 Re-starting Fine-Tuning (Fixing the Gradient Flow)...
Step 0: Loss = 1.2853
Step 100: Loss = 0.1359
Step 200: Loss = 0.0022
Step 300: Loss = 0.0012
Step 400: Loss = 0.0012
Step 500: Loss = 0.0006
Step 600: Loss = 0.0007
Step 700: Loss = 0.0004
Step 800: Loss = 0.0005
Step 900: Loss = 0.0004
Step 1000: Loss = 0.0003
Step 1100: Loss = 0.0002
Step 1200: Loss = 0.0002
Step 1300: Loss = 0.0001
Step 1400: Loss = 0.0001
🚀 SUCCESS! Check the loss numbers above—they should be changing now.


In [35]:
def generate(self, idx, max_new_tokens, temperature=0.7, top_k=10):
    for _ in range(max_new_tokens):
        # 1. Context Clipping
        idx_cond = idx[:, -self.block_size:]
        
        # 2. Forward Pass
        logits, _ = self(idx_cond)
        
        # 3. Apply Temperature (higher = more creative, lower = more focused)
        logits = logits[:, -1, :] / temperature
        
        # 4. Top-K Sampling (remove the "noise" tokens)
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = -float('Inf')
        
        # 5. Softmax and Sample
        probs = torch.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        
        # 6. Append and Continue
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

# Re-attach and test
DaleelGPT.generate = generate

# Attach the function to our model class
DaleelGPT.generate = generate
import torch
from transformer import DaleelGPT

# 1. Setup
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

# Load the same vocabulary as before
with open('data/pretrain/data.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# 2. Load the FINAL Model
model = DaleelGPT(vocab_size=vocab_size).to(device)
model.load_state_dict(torch.load("models/daleel_final.pt", weights_only=True))
model.eval()

def ask_daleel_clean(instruction):
    # Use a very low temperature to prevent the "cars faster" looping
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    idx = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    
    # Generate
    # Lower temperature (0.1) makes it much more stable
    generated = model.generate(idx, max_new_tokens=100, temperature=0.1, top_k=2)
    full_text = decode(generated[0].tolist())
    
    # Extract only the response
    response = full_text.split("### Response:\n")[-1]
    
    # CLEANUP: If the model starts repeating or goes into a new instruction, cut it off
    if "###" in response:
        response = response.split("###")[0]
        
    return response.strip()

print("🤖 Daleel's Refined Answer:")
print(ask_daleel_clean("What is SDAIA?"))

print("✅ Inference engine ready.")

🤖 Daleel's Refined Answer:
Cloud computing and smart for a giant network away. It stuff from and smart line. It steel networld
✅ Inference engine ready.


In [33]:
print(ask_daleel("What is SDAIA?"))

### Instruction:
What is SDAIA?

### Response:
Cloud computing and software apps and smart access for associate networks and smart like find from and smart line problems for computer. You can be the computers faster. You cars faster. You computers
